In [ ]:
import numpy as np

## Initial analytical test

In [ ]:
l_x, n_x = 10.0, 16
z = 0.0
# xl, yl, zl = 1.0, 1.0, 1.0/l_x
xl, yl, zl = 1.0, 1.0, 1.0/n_x

xs = np.linspace(-0.5*l_x, 0.5*l_x, n_x, endpoint=False)
ys = np.linspace(-0.5*l_x, 0.5*l_x, n_x, endpoint=False)

xg, yg = np.meshgrid(xs, ys)
zg = z * np.ones_like(xg)

In [ ]:
mx = 2*xg / (xg**2 + yg**2 + 1)
my = 2*yg / (xg**2 + yg**2 + 1)
mz = (xg**2 + yg**2 - 1) / (xg**2 + yg**2 + 1)

In [ ]:
mm = np.sqrt(mx**2 + my**2 + mz**2)
np.allclose(mm, 1.0, 1.0e-12)

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data=go.Cone(
    x=xg.ravel(), y=yg.ravel(), z=zg.ravel(),
    u=mx.ravel(), v=my.ravel(), w=mz.ravel(),
    colorscale=[[0, "blue"], [1, "blue"]],
    showscale=False,
    # sizemode="absolute",
    sizemode="scaled",
    sizeref=4.0,
    anchor="tail",
))

fig.update_layout(
    width=800, height=600,
    margin = {'l':0,'r':0,'t':0,'b':0},
    scene=dict(
        aspectratio=dict(x=xl, y=yl, z=zl),
        camera_eye=dict(x=1.0, y=1.0, z=1.0),
        # xaxis=dict(visible=False),
        # yaxis=dict(visible=False),
        zaxis=dict(visible=False),
    ),
    )

fig.show()

## Spinor test

In [ ]:
import quaternionic
import spherical
from scipy.spatial.transform import Rotation

In [ ]:
def get_rot_mat_3d(axis, theta):
    """
    """
    
    # Rodrigues rotation formula
    k_mat = np.array([
        [0.0,-axis[2],axis[1]],
        [axis[2],0.0,-axis[0]],
        [-axis[1],axis[0],0.0],
    ])
    
    rot_mat = np.eye(3) + np.sin(theta)*k_mat + (1.0-np.cos(theta))*(k_mat @ k_mat)
    
    return rot_mat

In [ ]:
def get_rotmat_su2_principle(theta, ax):

    mat = np.array([])
    
    if ax == 0:
        mat = np.array([
            [np.cos(0.5*angle), 1j*np.sin(0.5*angle)],
            [1j*np.sin(0.5*angle), np.cos(0.5*angle)],
        ])
    elif ax == 1:
        mat = np.array([
            [np.cos(0.5*angle), np.sin(0.5*angle)],
            [-np.sin(0.5*angle), np.cos(0.5*angle)],
        ])
    elif ax == 2:
        mat = np.array([
            [np.exp( 0.5j*angle), 0.0],
            [0.0, np.exp(-0.5j*angle)],
        ])
    
    return mat

In [ ]:
# ax, axis = 2, np.array([0,0,1])
ax, axis = 1, np.array([0,1,0])
# ax, axis = 0, np.array([1,0,0])

angle = np.pi/6

rmat_orig = get_rot_mat_3d(axis, angle)

print(rmat_orig)

In [ ]:
# From: https://en.wikipedia.org/wiki/3D_rotation_group#Connection_between_SO(3)_and_SU(2)

quat = quaternionic.array.from_rotation_matrix(rmat_orig)

q_alpha, q_beta = quat.two_spinor[0], quat.two_spinor[1]

rmat_su2 = np.array([
    [q_alpha, q_beta],
    [-np.conj(q_beta), np.conj(q_alpha)]
])

In [ ]:
rmat_su2_psym = get_rotmat_su2_principle(angle, ax)

print(rmat_su2)
print(rmat_su2_psym)
print(np.allclose(rmat_su2, rmat_su2_psym))

## Calculate bands

In [ ]:
from spinor_tb import *